# HWIN-Bench v1.0 — Results Analysis

Quick analysis notebook for canonical benchmark results.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Load canonical results
results_path = Path('../../benchmark/results/grqa_recomputation/all_results.json')

if not results_path.exists():
    print(f"Results not found at {results_path}")
    print("The canonical reference results ship with the repository at")
    print("benchmark/results/grqa_recomputation/")
else:
    with open(results_path) as f:
        results = json.load(f)
    variables = {v: d for v, d in results.items()}
    print(f"Loaded results for {len(variables)} variables")


In [ ]:
# Extract metric summaries
# Canonical run schema: {var: {model: {metrics: {r2: {mean, ci_95, n_folds}, ...}, total_time, ...}}}
summary_rows = []

for var_id, var_entry in variables.items():
    for model_name, model_results in var_entry.items():
        metrics = model_results['metrics']
        summary_rows.append({
            'variable': var_id,
            'model': model_name,
            'r2_mean': metrics['r2']['mean'],
            'r2_ci_lower': metrics['r2']['ci_95'][0],
            'r2_ci_upper': metrics['r2']['ci_95'][1],
            'rmse_mean': metrics['rmse']['mean'],
            'mae_mean': metrics['mae']['mean'],
            'mape_mean': metrics['mape']['mean'],
            'medae_mean': metrics['medae']['mean'],
            'n_folds': metrics['r2']['n_folds'],
            'total_time': model_results['total_time']
        })

df = pd.DataFrame(summary_rows)
print(f"Summary shape: {df.shape}")
df.head()


In [ ]:
# Best model per variable by R²
best_models = df.loc[df.groupby('variable')['r2_mean'].idxmax()]
print("Best model per variable (by R²):")
print(best_models[['variable', 'model', 'r2_mean', 'rmse_mean']].to_string(index=False))

In [ ]:
# R² heatmap: variables × models
pivot_r2 = df.pivot(index='variable', columns='model', values='r2_mean')

plt.figure(figsize=(14, 10))
sns.heatmap(pivot_r2, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5,
            cbar_kws={'label': 'R² (mean across recorded folds)'})
plt.title('HWIN-Bench v1.0 — R² by Variable and Model')
plt.tight_layout()
plt.show()

In [ ]:
# RMSE heatmap
pivot_rmse = df.pivot(index='variable', columns='model', values='rmse_mean')

plt.figure(figsize=(14, 10))
sns.heatmap(pivot_rmse, annot=True, fmt='.3f', cmap='RdYlGn_r',
            cbar_kws={'label': 'RMSE (mean across recorded folds)'})
plt.title('HWIN-Bench v1.0 — RMSE by Variable and Model')
plt.tight_layout()
plt.show()

In [ ]:
# Model ranking across all variables
model_ranking = df.groupby('model').agg({
    'r2_mean': ['mean', 'std', 'count'],
    'rmse_mean': ['mean', 'std'],
    'total_time': 'mean'
}).round(4)

model_ranking.columns = ['_'.join(col).strip() for col in model_ranking.columns.values]
model_ranking = model_ranking.sort_values('r2_mean_mean', ascending=False)
print("Model ranking (average R² across variables):")
print(model_ranking[['r2_mean_mean', 'r2_mean_std', 'rmse_mean_mean', 'total_time_mean']].to_string())

In [ ]:
# Per-variable detailed view
variable = 'HWIN-VAR-001'  # Water Temperature
var_df = df[df['variable'] == variable].sort_values('r2_mean', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(var_df['model'], var_df['r2_mean'],
         xerr=(var_df['r2_ci_upper'] - var_df['r2_ci_lower']) / 2, capsize=5)
plt.xlabel('R2 (mean, 95% CI across folds)')
plt.title(f'{variable} - Water Temperature: Model Performance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Training time comparison
time_df = df.groupby('model')['total_time'].mean().sort_values()

plt.figure(figsize=(10, 6))
time_df.plot(kind='barh')
plt.xlabel('Average Total Training Time (seconds)')
plt.title('Model Training Time Comparison')
plt.tight_layout()
plt.show()

## Statistical Test Results

Load and visualize statistical test outputs (Kruskal-Wallis, Wilcoxon, Cliff's Delta, Holm-Bonferroni).

In [ ]:
# Load statistical test results if available
stats_path = Path('../../benchmark/results/statistical_test_results.json')

if stats_path.exists():
    with open(stats_path) as f:
        stats_results = json.load(f)
    print("Statistical test results loaded")
    print(json.dumps(stats_results, indent=2)[:2000])
else:
    print("Statistical test results not found. Run full benchmark to generate.")